# Variables separables

In [214]:
import sympy as sp
import numpy as np
import plotly.graph_objects as go

# Funciones compartidas

In [215]:
# Analizar la ecuación diferencial
def analizar_ecuacion_diferencial(ecuacion, y):
    print("Ecuación diferencial planteada:")
    display(ecuacion)

    clasificacion = sp.classify_ode(ecuacion, y)

    # Mostrar las caracteristicas de la ecuación diferencial
    print("Clasificaciones de la ecuación diferencial:")

    es_lineal = any('linear' in c for c in clasificacion)
    print(f"Es lineal: {es_lineal}")

    es_ordinaria = any('ordinary' in c for c in clasificacion)
    print(f"Es ordinaria: {es_ordinaria}")

    orden = sp.ode_order(ecuacion, y)
    print(f"Orden: {orden}")

    print("Posibles métodos de solución:")
    for metodo in clasificacion:
        print(f"- {metodo}")

In [216]:
def solucion_particular(solucion_general, y_0, constante):
    # Extraer la constante de integración C1 de la solución general
    C1 = sp.symbols(constante)
    
    # Resolver para C1 usando la condición inicial y(0) = y_0
    condicion_inicial = solucion_general.subs({x: 0, y: y_0})
    C1_valor = sp.solve(condicion_inicial, C1)[0]
    
    # Sustituir el valor de C1 en la solución general para obtener la solución particular
    solucion_particular = solucion_general.subs(C1, C1_valor)
    
    return solucion_particular

In [217]:
def graficar_solucion(solucion, x_min, x_max, num_puntos=100):
    # Convertir la solución simbólica a una función numérica graficar
    latex_equation = f"${sp.latex(solucion)}$"

    # Crear un rango de valores de x
    x_vals = np.linspace(x_min, x_max, num_puntos)
    
    # Evaluar la solución para cada valor de x
    y_vals = []
    for val_x in x_vals:
        valor_numerico = float(solucion.rhs.subs(x, val_x).evalf())
        y_vals.append(valor_numerico)
    
    # Crear la figura de Plotly
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=x_vals, 
        y=y_vals, 
        mode='lines', 
        line=dict(color='firebrick', width=3)
    ))
    
    # Configurar el diseño del gráfico
    fig.update_layout(
        title=f'Solución Particular: {latex_equation}',
        xaxis_title='x',
        yaxis_title='y',
        showlegend=False,
        template='plotly_white'
    )
    
    return fig

# Opcion 1: Usar dsolve (differential solve) para encontrar la función general automáticamente

In [218]:
# Definir la variable simbólica y la función dependiente
x = sp.Symbol('x')
y = sp.Function('y')(x)

In [219]:
# sp.Derivative(y, x) representa la derivada de primer orden dy/dx
derivada = sp.Derivative(y, x)

In [220]:
# Construir la ecuación diferencial
ecuacion = sp.Eq(derivada, (2 * x) / (3 * y ** 2))
print("Ecuación diferencial planteada:")
display(ecuacion)

Ecuación diferencial planteada:


Eq(Derivative(y(x), x), 2*x/(3*y(x)**2))

In [221]:
# Mostrar las características de la ecuación diferencial
analizar_ecuacion_diferencial(ecuacion, y)

Ecuación diferencial planteada:


Eq(Derivative(y(x), x), 2*x/(3*y(x)**2))

Clasificaciones de la ecuación diferencial:
Es lineal: False
Es ordinaria: False
Orden: 1
Posibles métodos de solución:
- separable
- 1st_exact
- Bernoulli
- 1st_power_series
- lie_group
- separable_Integral
- 1st_exact_Integral
- Bernoulli_Integral


In [225]:
# dsolve (differential solve) encuentra la función general automáticamente
solucion_general = sp.dsolve(ecuacion, y)[0]

print("La solución general de la ecuación diferencial es:")
display(solucion_general)

La solución general de la ecuación diferencial es:


Eq(y(x), (C1 + x**2)**(1/3))

In [226]:
y_0 = 1  # Condición inicial y(0) = 1
sol_particular = solucion_particular(solucion_general, y_0=y_0, constante='C1')
print(f"Solución particular con y(0) = {y_0}:")
display(sol_particular)

Solución particular con y(0) = 1:


Eq(y(x), (x**2 + 1)**(1/3))

In [229]:
grafica = graficar_solucion(sol_particular, x_min=0, x_max=50)
grafica.show()

# Opcion 2: Resolver paso a paso usando integración haciendo la integracion manualmente

In [231]:
# Definir las variables simbólicas y la constante de integración
x = sp.Symbol('x')
y = sp.Symbol('y')
C = sp.Symbol('C1')

In [232]:
# Hacer el despeje manualmente y definirlo
integrando_izquierdo = 3 * y ** 2
integrando_derecho = 2 * x

print("Integrales planteadas:")
print("Lado izquierdo (dy):")
display(integrando_izquierdo)
print("Lado derecho (dx):")
display(integrando_derecho)

Integrales planteadas:
Lado izquierdo (dy):


3*y**2

Lado derecho (dx):


2*x

In [233]:
# Integrar ambos lados de la ecuación
integral_izq = sp.integrate(integrando_izquierdo, y)
integral_der = sp.integrate(integrando_derecho, x)

print("Resultado de la integración:")
print("Integral izquierda:")
display(integral_izq)
print("Integral derecha:")
display(integral_der)

Resultado de la integración:
Integral izquierda:


y**3

Integral derecha:


x**2

In [234]:
# Plantear la ecuacion despues de integrar y sumar la constante de integración C
ecuacion_implicita = sp.Eq(integral_izq, integral_der + C)

print("Ecuación despues de integrar:")
display(ecuacion_implicita)

Ecuación despues de integrar:


Eq(y**3, C1 + x**2)

In [235]:
# Despejar 'y' para obtener la solución general
soluciones = sp.solve(ecuacion_implicita, y)
solucion_general = sp.Eq(y, soluciones[0])

print("La solución general de la ecuación diferencial es:")
display(solucion_general)

La solución general de la ecuación diferencial es:


Eq(y, (C1 + x**2)**(1/3))

In [236]:
y_0 = 1  # Condición inicial y(0) = 1
sol_particular = solucion_particular(solucion_general, y_0=y_0, constante='C1')
print(f"Solución particular con y(0) = {y_0}:")
display(sol_particular)

Solución particular con y(0) = 1:


Eq(y, (x**2 + 1)**(1/3))

In [238]:
grafica = graficar_solucion(sol_particular, x_min=0, x_max=50)
grafica.show()

# Opcion 3: separar variables usando SymPy

In [241]:
# Definir la variable simbólica y la función dependiente
x = sp.Symbol('x')
y = sp.Function('y')(x)

In [242]:
# Construir la ecuación diferencial
derivada = sp.Derivative(y, x)

In [243]:
# sp.Eq() construye la igualdad matemática: Eq(lado_izquierdo, lado_derecho)
ecuacion = sp.Eq(derivada, (2 * x) / (3 * y ** 2))
print("Ecuación diferencial planteada:")
display(ecuacion)

Ecuación diferencial planteada:


Eq(Derivative(y(x), x), 2*x/(3*y(x)**2))

In [244]:
# Mostrar las características de la ecuación diferencial
analizar_ecuacion_diferencial(ecuacion, y)

Ecuación diferencial planteada:


Eq(Derivative(y(x), x), 2*x/(3*y(x)**2))

Clasificaciones de la ecuación diferencial:
Es lineal: False
Es ordinaria: False
Orden: 1
Posibles métodos de solución:
- separable
- 1st_exact
- Bernoulli
- 1st_power_series
- lie_group
- separable_Integral
- 1st_exact_Integral
- Bernoulli_Integral


In [245]:
# Separar variables usando SymPy
ecuacion_separada = sp.dsolve(ecuacion, y, hint='separable_Integral')
print("Ecuación diferencial separada:")
display(ecuacion_separada)

Ecuación diferencial separada:


Eq(Integral(_y**2, (_y, y(x))), C1 + Integral(2*x/3, x))

In [246]:
# Resolver las integrales
integral_resuelta = sp.Eq(ecuacion_separada.lhs.doit(), ecuacion_separada.rhs.doit())

print("Ecuación diferencial después de resolver las integrales:")
display(integral_resuelta)

Ecuación diferencial después de resolver las integrales:


Eq(y(x)**3/3, C1 + x**2/3)

In [247]:
# Despejar 'y' para obtener la solución general
soluciones = sp.solve(integral_resuelta, y)
solucion_general = sp.Eq(y, soluciones[0])

print("La solución general de la ecuación diferencial es:")
display(solucion_general)

La solución general de la ecuación diferencial es:


Eq(y(x), (3*C1 + x**2)**(1/3))

In [248]:
y_0 = 1  # Condición inicial y(0) = 1
sol_particular = solucion_particular(solucion_general, y_0=y_0, constante='C1')
print(f"Solución particular con y(0) = {y_0}:")
display(sol_particular)

Solución particular con y(0) = 1:


Eq(y(x), (x**2 + 1)**(1/3))

In [249]:
grafica = graficar_solucion(sol_particular, x_min=0, x_max=50)
grafica.show()